In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, "..")
from support.paths import resolve

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
TARGET               = os.environ.get("TARGET", "NSW")
TARGET_LOWER         = TARGET.lower()
FEATURE_DATASET_STEM = Path(os.environ.get("FEATURE_DATASET", "1_dispatch_price.parquet")).stem
PRICE_TRANSFORM_SCALE = 100.0

In [ ]:
# ── Cross-feature computation ──────────────────────────────────────────────────
# Produces 4 columns used by 4_Model/2_train.ipynb that are not model inputs
# but are needed for the training pipeline (naive blend, clip threshold, etc.).
# Computed once here and stored alongside the feature data.

def compute_cross_feats(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with 4 auxiliary cross-feature columns.

    Columns
    -------
    doy_sin            : 365.25-day annual sinusoidal cycle
    doy_cos            : 365.25-day annual cosine cycle
    sa_spread_live     : arcsinh((SA - NSW) / scale) — SA-NSW interconnector pressure
    region_spike_score : count of QLD/VIC/SA regions with price > 150 (0–3)
    """
    n   = len(df)
    doy = df.index.day_of_year.astype(np.float32)

    doy_sin = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    doy_cos = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

    if "sa_price" in df.columns and "nsw_price" in df.columns:
        sa  = df["sa_price"].fillna(0.0).values.astype(np.float32)
        nsw = df["nsw_price"].fillna(0.0).values.astype(np.float32)
        sa_spread = np.arcsinh((sa - nsw) / PRICE_TRANSFORM_SCALE).clip(-10, 10).astype(np.float32)
    elif "nsw_price_vs_sa_price_spread" in df.columns:
        sa_spread = np.arcsinh(
            df["nsw_price_vs_sa_price_spread"].fillna(0.0).values / PRICE_TRANSFORM_SCALE
        ).clip(-10, 10).astype(np.float32)
    else:
        sa_spread = np.zeros(n, dtype=np.float32)

    score = np.zeros(n, dtype=np.float32)
    for reg_col in ["qld_price", "vic_price", "sa_price"]:
        if reg_col in df.columns:
            score += (df[reg_col].fillna(0.0).values > 150).astype(np.float32)

    return pd.DataFrame({
        "doy_sin":            doy_sin,
        "doy_cos":            doy_cos,
        "sa_spread_live":     sa_spread,
        "region_spike_score": score,
    }, index=df.index)

In [ ]:
# ── Build and save ─────────────────────────────────────────────────────────────
_feature_path = resolve(f"2_Features_build/Feature_data/{FEATURE_DATASET_STEM}.parquet")
df = pd.read_parquet(_feature_path)

cross = compute_cross_feats(df)

# Price col (clip threshold source) + naive baseline lags (blend baseline).
_price_lag_cols = [
    f"{TARGET_LOWER}_price",
    f"{TARGET_LOWER}_price_lag_336",
    f"{TARGET_LOWER}_price_lag_48",
]
_price_lag_cols = [c for c in _price_lag_cols if c in df.columns]

auxiliary = pd.concat([cross, df[_price_lag_cols]], axis=1)

_out_path = resolve(f"2_Features_build/Feature_data/{TARGET}_auxiliary_{FEATURE_DATASET_STEM}.parquet")
auxiliary.to_parquet(_out_path)
print(f"Saved {auxiliary.shape[1]} columns × {len(auxiliary):,} rows → {_out_path}")
print(auxiliary.dtypes)